In [ ]:
import csv
import json

from scielo_scholarly_data import standardizer

In [ ]:
base_to_keys = {
    'doaj': {
        'titles': ['Journal title', 'Alternative title'],
        'countries': ['Country of publisher', 'Country of other organisation'],
        'issns': ['Journal ISSN (print version)', 'Journal EISSN (online version)'],
        'delimiter': ',',
    }, 
    'latindex': {
        'titles': ['tit_propio',],
        'countries': ['nombre_largo'],
        'issns': ['issn_e', 'issn_l', 'issn_imp'],
        'delimiter': ';',
    },
    'mug_study_brazil': {
        'titles': ['"Título"', '"Outro Título"'],
        'countries': ['"País"',],
        'issns': ['"ISSN"',],
        'issn_list': ['"Outro(s) ISSN(s)"'],
        'issn_list_delimiter': '',
        'delimiter': ',',
        'quoting': csv.QUOTE_ALL,
    },
    'mug_study_spain': {
        'titles': ['"Título"', '"Outro Título"'],
        'countries': ['"País"',],
        'issns': ['"ISSN"',],
        'issn_list': ['"Outro(s) ISSN(s)"'],
        'issn_list_delimiter': '',
        'delimiter': ',',
        'quoting': csv.QUOTE_ALL,
    },
    'scielo': {
        'titles': ['"title at SciELO"', '"title + subtitle SciELO"', '"short title SciELO"', '"short title ISO"', '"title PubMed"'],
        'countries': ['"collection"'],
        'issns': ['"ISSN SciELO"'],
        'issn_list': ['"ISSN\'s"',],
        'issn_list_delimiter': ';',
        'delimiter': ',',
        'quoting': csv.QUOTE_ALL,
    },
    'scimagojr': {
        'titles': ['Title',],
        'countries': ['Country'],
        'issns': [],
        'issn_list': ['Issn',],
        'issn_list_delimiter': ', ',
        'delimiter': ';'
    },
    'scopus_sources': {
        'titles': ['Source Title', 'Medline-sourced Title? (See additional details under separate tab.)', 'Related title 1', 'Other related title 2', 'Other related title 3', 'Other related title 4'],
        'countries': [],
        'issns': ['Print-ISSN', 'E-ISSN'],
        'delimiter': ','
    },
    'scopus_accepted': {
        'titles': ['Title name'],
        'countries': [],
        'issns': ['Print ISSN', 'EISSN'],
        'delimiter': ','
    },
    'ulrich': {
        'titles': ['"title_title"'],
        'countries': ['"title_country"'],
        'issns': ['"title_issn"'],
        'delimiter': ',',
        "quoting": csv.QUOTE_ALL,
    },
    'wos_extra': {
        'titles': ['"Title"'],
        'countries': ['"Country / Region"'],
        'issns': ['"uISSN"',],
        'issn_list': ['"ISSN/e-ISSN"'],
        'issn_list_delimiter': ' / ',
        'delimiter': ','
    },
    'wos_jcr': {
        'titles': ['title', 'title20'],
        'countries': ['country'],
        'issns': ['eISSN', 'ISSN'],
        'delimiter': ',',
    },
    'mug_isis_extra': {
        'titles': ['title 1', 'title 2'],
        'countries': [],
        'issns': ['issn',],
        'delimiter': '|',
    },
    # special cases
    'nlm': {
        'titles': ['Title Abbreviation', 'Title(s)'],
        'countries': ['Country of Publication'],
        'issns': ['ISSN',],
        'delimiter': '',
    },
    'portal_issn_2024': {
        'titles': ['"key_title"', '"main_title"'],
        'countries': ['"country"'],
        'issns': ['"issn_l"', '"issn"'],
        'delimiter': ',',
    },
    'portal_issn_2019': {
        'title_list': ['key_title', 'title_proper', 'abbreviated_key_title', 'caption_title', 'other_variant_title'],
        'country_list': ['country'],
        'issn_list': ['issn', 'issn_l'],
        'delimiter': ',',
    },
}

def stz_data(titles, countries, issns):
    titles = list(set([t for t in [standardizer.journal_title_for_deduplication(t).upper() for t in titles] if t != '']))
    countries = list(set([c for c in [standardizer.document_title_for_deduplication(c).upper() for c in countries] if c != '']))
    issns = list(set([i for i in [standardizer.journal_issn(i) for i in issns if i != ''] if i is not None]))
    return {'titles': titles, 'countries': countries, 'issns': issns}

def load_base(path, name, debug=False):
    title_keys = base_to_keys[name]['titles']
    country_keys = base_to_keys[name]['countries']
    issn_keys = base_to_keys[name]['issns']
    issn_list_keys = base_to_keys[name].get('issn_list')
    issn_list_delimiter = base_to_keys[name].get('issn_list_delimiter')
    delimiter = base_to_keys[name]['delimiter']
    quoting = base_to_keys[name].get('quoting', csv.QUOTE_MINIMAL)

    data = []
    if debug:
        issns_len = {}
        issn_list_len = 0

    with open(path) as fin:
        for row in csv.DictReader(fin, fieldnames=fin.readline().strip().split(delimiter), delimiter=delimiter, quoting=quoting):
            titles = [row[t] for t in title_keys]
            countries = [row[c] for c in country_keys]
            issns = [row[i] for i in issn_keys]

            if debug:
                for ik in issn_keys:
                    if ik not in issns_len:
                        issns_len[ik] = 0

                    if len(row[ik]) > issns_len[ik]:
                        issns_len[ik] = len(row[ik])
                        print(f'DEBUG. Field issns {ik} contains {issns_len[ik]} chars.')

            if issn_list_keys is not None and issn_list_delimiter is not None:
                if issn_list_delimiter != '':
                    for k in issn_list_keys:
                        for i in row[k].split(issn_list_delimiter):
                            issns.append(i)
                        
                        if debug:
                            if issn_list_len < len(row[k]):
                                issn_list_len = len(row[k])
                                print(f'DEBUG. Field issn_list contains {issn_list_len} chars.')
                else:
                    for k in issn_list_keys:
                        k_size = len(row[k])
                        for ks in range(0, k_size, 9):
                            issns.append(row[k][ks:ks+9])

            data.append(stz_data(titles, countries, issns))
            
    return data

def load_base_nlm(path):
    data = []

    def parse_entry(entry):
        titles = []
        countries = []
        issns = []

        for i in entry.split('\n'):
            if i.startswith('Title Abbr'):
                titles.append(i.split(' Title Abbreviation: ')[-1])

            if i.startswith('Title(s)'):
                titles.append(i.split('Title(s): ')[-1])

            if i.startswith('Country o'):
                countries.append(i.split('Country of Publication: ')[-1])

            if i.startswith('ISSN'):
                temp = [x.strip() for x in i.split('ISSN:')]
                if len(temp) > 0:
                    for x in temp:
                        for xi in x.split('; '):
                            issns.append(xi[0:9])

        return stz_data(titles, countries, issns)

    def parse_text(text):
        entries = text.strip().split('\n\n')
        return [parse_entry(entry) for entry in entries]

    def read_file(file_path):
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()

    text = read_file(path)
    parsed_data = parse_text(text)
    for entry in parsed_data:
        data.append(entry)

    return data

def load_base_portal_issn(path):
    data = []

    title_keys = base_to_keys['portal_issn_2019']['title_list']
    country_keys = base_to_keys['portal_issn_2019']['country_list']
    issn_keys = base_to_keys['portal_issn_2019']['issn_list']

    for line in open(path):
        row = json.loads(line)
        titles = []
        countries = []
        issns = []

        for tk in title_keys:
            titles.extend(row.get(tk, []))

        for ck in country_keys:
            countries.extend(row.get(ck, []))

        for ik in issn_keys:
            issns.extend(row.get(ik, []))

        data.append(stz_data(titles, countries, issns))

    return data

In [ ]:
portal_issn_2019 = load_base_portal_issn('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/portal_issn_2019.json')
portal_issn_2019[0:3]

In [ ]:
portal_issn_2024 = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/portal_issn_2024.csv', 'portal_issn_2024', debug=True)
portal_issn_2024[0:3]

In [ ]:
doaj = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/doaj.csv', 'doaj', debug=True)
doaj[0:3]

In [ ]:
latindex = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/latindex.csv', 'latindex', debug=True)
latindex[0:3]

In [ ]:
ms_brazil = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/mug_brazil.csv', 'mug_study_brazil', debug=True)
ms_brazil[0:3]

In [ ]:
ms_spain = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/mug_spain.csv', 'mug_study_spain', debug=True)
ms_spain[0:3]

In [ ]:
scielo = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/scielo.csv', 'scielo', debug=True)
scielo[0:3]

In [ ]:
scimagojr = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/scimagojr.csv', 'scimagojr', debug=True)
scimagojr[0:3]

In [ ]:
scopus_sources = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/scopus_sources.csv', 'scopus_sources', debug=True)
scopus_sources[0:3]

In [ ]:
scopus_accepted = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/scopus_accepted.csv', 'scopus_accepted', debug=True)
scopus_accepted[0:3]

In [ ]:
ulrich = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/ulrich.csv', 'ulrich', debug=True)
ulrich[0:3]

In [ ]:
wos_extra = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/wos_extra.csv', 'wos_extra', debug=True)
wos_extra[0:3]

In [ ]:
wos_jcr = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/wos_jcr.csv', 'wos_jcr', debug=True)
wos_jcr[0:3]

In [ ]:
mug_isis_extra = load_base('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/mug_isis_extra.seq', name='mug_isis_extra', debug=True)
mug_isis_extra[0:3]

In [ ]:
nlm = load_base_nlm('/home/rafaeljpd/Data/cimetrias/2024-liinc/v0.8/cleaned/nlm.txt')
nlm[0:3]

In [ ]:
data = []
number_of_issn_fields = 0
for b in [portal_issn_2019, portal_issn_2024, doaj, latindex, ms_brazil, ms_spain, scielo, scimagojr, scopus_sources, scopus_accepted, ulrich, wos_extra, wos_jcr, mug_isis_extra, nlm]:
	for row in b:
		issns = []
		r_titles = row['titles']
		r_countries = row['countries']
		r_issns = row['issns']
		for i in r_issns:
			if len(i) == 9:
				issns.append(i)
				
		if len(issns) > number_of_issn_fields:
			number_of_issn_fields = len(issns)

		if len(issns) > 0:
			data.append(r_issns)

In [ ]:
with open('gold.raw.csv') as fout:
    fout.write('issn1,issn2,issn3,issn4\n')

with open('gold.csv', 'w') as fout:
    for d in data:
        while len(d) < number_of_issn_fields:
            d.append('')
        fout.write(','.join(d) + '\n')

In [ ]:
! sort --unique gold.raw.csv >> gold.csv